# 清洗后数据复核：文本保留与验证标签争议

## 摘要

本 Notebook 对清洗后的两个集合做**只读全量来源对账**，并核验本轮逐条语义复核所引用的真实文本和标签。下方统计由运行时重新计算；不会修改 CSV、原始报告或复核决定。

语义结论属于助手复核意见，尚未应用为标签修改。机械核验通过不能等同于全部标签正确。

## 背景与方法

- 单位：一个集合中的一条评论；以 `(split, id)` 和原始记录号对照。
- 基线：`data/raw`；检查对象：`data/processed` 与 `data/quarantine`。
- 实现：`scripts/audit_processed_dataset.py` 的只读 API，不调用清洗函数，也不以旧校验结果代替重新比较。
- 语义证据：`data/reports/processed_review` 中的三份 JSON，涵盖全部已编辑文本以及指定的两条验证样本。

### 关键假设

标签域为 `-2=未提及、-1=负面、0=中性、1=正面`。同文训练标签仅用于定位分歧，不作为真值。缺少官方细化标注规范时，边界判断保留候选；不擅自改动固定验证集。

运行环境需要 Python、nbformat 和 ipykernel；显示表格使用 IPython。可从项目根或 notebooks 目录启动。

**本次保存输出的执行方式：**独立 Jupyter kernel 启动尝试未完成，已中止；随后以同一 Python 进程的标准 `exec` 按顺序真实执行全部代码单元，并捕获标准输出和 IPython 富文本表格。所有断言均实际运行。该 Notebook 仍可在正常 Jupyter 环境从上到下重跑。

In [1]:
# 从当前目录向上查找项目入口，避免在笔记本中写死个人机器路径。
# 本单元只准备只读函数和展示工具；不会执行清洗或覆盖现有复核证据。
from pathlib import Path
from collections import Counter
import csv
import hashlib
import html
import json
import sys
from IPython.display import display, HTML, Markdown

start = Path.cwd().resolve()
root = next((p for p in (start, *start.parents)
             if (p / "scripts/audit_processed_dataset.py").is_file()), None)
assert root is not None, "请从本项目目录内启动 Notebook。"
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from scripts.audit_processed_dataset import audit_processed_dataset
from scripts.label_config import ASPECT_COLUMNS, CLASS_VALUES
from scripts.validate_dataset import file_sha256
review_dir = root / "data/reports/processed_review"

def read_json(name):
    """读取已保存的复核证据，供本次重新核验和展示，不生成或覆盖报告。"""
    return json.loads((review_dir / name).read_text(encoding="utf-8"))

def text_sha(text):
    """对 CSV 解析后的评论内容计算摘要，排除 CSV 外层转义格式的影响。"""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def show_table(headers, rows):
    """将证据表格转义后展示；评论中的 HTML 字符只按文本呈现。"""
    def cell(value):
        return html.escape(str(value)).replace("\n", "<br>")
    body = "".join("<tr>" + "".join("<td>" + cell(v) + "</td>" for v in row) + "</tr>" for row in rows)
    head = "<tr>" + "".join("<th>" + cell(h) + "</th>" for h in headers) + "</tr>"
    display(HTML('<div style="overflow-x:auto"><table style="border-collapse:collapse;text-align:left;max-width:1100px">'
                 + "<thead>" + head + "</thead><tbody>" + body + "</tbody></table></div>"))
print("项目根目录：", root)


项目根目录： D:\code\granu_review


## 数据核验

### 1. 全量重新读取与条数对账

逐条比较保留记录的 ID、顺序、20 个标签和文本，核对每个删除字符及隔离副本，并重新检查凭据中的文件摘要。此函数只读，不写出新的数据或报告。

In [2]:
# 重新读取原始、处理后和隔离 CSV 做独立对账，再与保存的统计逐项比较。
# 保存的“通过”状态只能作为对照，不能替代本次逐条正文和标签检查。
audit, edit_evidence, dispute_evidence = audit_processed_dataset(root)
assert audit["mechanical_checks_passed"]
assert audit["unplanned_text_edits"] == 0
assert audit["changed_label_cells"] == 0
saved = read_json("lineage_audit.json")
for key in ("raw_counts", "retained_counts", "unchanged_text_counts", "edited_text_records",
            "removed_characters", "removed_character_count", "label_cells_compared",
            "quarantined_records_preserved_exactly", "application_issue_ids_reconciled"):
    assert audit[key] == saved[key], (key, "保存的复核结果已不适用")
edit_counts = Counter(record["split"] for record in edit_evidence)
show_table(["集合", "原始评论数", "清洗后评论数", "文本原样保留", "编辑文本数"],
           [(split, audit["raw_counts"][split], audit["retained_counts"][split],
             audit["unchanged_text_counts"][split], edit_counts[split])
            for split in ("train", "validation")])
show_table(["核验项", "本次复算结果"], [
    ("保留记录逐标签比较单元格", audit["label_cells_compared"]),
    ("计划外文本改动", audit["unplanned_text_edits"]),
    ("被修改的标签", audit["changed_label_cells"]),
    ("完整保留原文的隔离记录", audit["quarantined_records_preserved_exactly"]),
    ("已核对应用日志", audit["application_issue_ids_reconciled"]),
    ("删除控制字符总数", audit["removed_character_count"]),
])


集合,原始评论数,清洗后评论数,文本原样保留,编辑文本数
train,105000,104969,104959,10
validation,15000,15000,14999,1


核验项,本次复算结果
保留记录逐标签比较单元格,2399380
计划外文本改动,0
被修改的标签,0
完整保留原文的隔离记录,31
已核对应用日志,28
删除控制字符总数,21


### 2. 将语义复核记录重新绑定到真实 CSV

独立读取原始与清洗 CSV 中被引用的记录，验证文本 SHA-256、文件摘要、全部引文、20 方面顺序、标签域，以及同组训练标签。语义判断本身不由这些断言证明；断言验证的是判断引用了正确证据。

In [3]:
# 将逐条语义意见重新绑定到实际 CSV，先验证来源摘要，再核对原文、标签及引文。
# 这里只在内存收集需要展示的样本；训练标签用于呈现分歧，不覆盖验证标签。
edit_review = read_json("text_edit_semantic_review.json")
label_reviews = [read_json("validation_13036_analysis.json"),
                 read_json("validation_14365_analysis.json")]
filenames = {"train": "sentiment_analysis_trainingset.csv",
             "validation": "sentiment_analysis_validationset.csv"}
references = {(r["split"], r["id"]) for r in edit_review["records"]}
references.update((r["split"], r["id"]) for r in label_reviews)
for review in label_reviews:
    if "training_reference" in review:
        references.add(("train", review["training_reference"]["id"]))
    for ref in review.get("training_counterparts", []):
        references.add(("train", ref["id"]))

selected = {}
for stage in ("raw", "processed"):
    for split, filename in filenames.items():
        with (root / "data" / stage / filename).open(encoding="utf-8-sig", newline="") as stream:
            reader = csv.DictReader(stream)
            assert reader.fieldnames == ["id", "content", *ASPECT_COLUMNS]
            for number, row in enumerate(reader, 1):
                if (split, row["id"]) in references:
                    key = stage, split, row["id"]
                    assert key not in selected, (key, "ID重复")
                    selected[key] = number, row

def actual(stage, split, identity):
    """按数据阶段、集合和 ID 读取内存样本，避免隔离后记录序号移动导致错配。"""
    return selected[(stage, split, identity)]

for source in edit_review["source_files"]:
    assert file_sha256(Path(source["path"])) == source["sha256"]
assert file_sha256(root / "data/reports/processed/processing_changes.jsonl") == edit_review["processing_changes_sha256"]
assert {(r["split"], r["id"]) for r in edit_review["records"]} == {(r["split"], r["id"]) for r in edit_evidence}

for review in edit_review["records"]:
    raw_number, raw = actual("raw", review["split"], review["id"])
    processed_number, processed = actual("processed", review["split"], review["id"])
    assert raw_number == review["raw_record_number"]
    assert processed_number == review["processed_record_number"]
    assert text_sha(raw["content"]) == review["raw_text_sha256"]
    assert text_sha(processed["content"]) == review["processed_text_sha256"]
    assert len(raw["content"]) == review["raw_length"]
    assert len(processed["content"]) == review["processed_length"]
    assert all(raw[aspect] == processed[aspect] for aspect in ASPECT_COLUMNS)
    deleted = Counter()
    for edit in review["edits"]:
        assert edit["operation"] == "delete"
        span = raw["content"][edit["raw_start"]:edit["raw_end_exclusive"]]
        assert [f"U+{ord(char):04X}" for char in span] == edit["deleted_code_points"]
        assert raw["content"][:edit["raw_start"]].endswith(edit["left_context"])
        assert raw["content"][edit["raw_end_exclusive"]:].startswith(edit["right_context"])
        assert edit["processed_context"] in processed["content"]
        deleted.update(f"U+{ord(char):04X}" for char in span)
    assert dict(deleted) == review["removed_code_point_counts"]
    assert sum(deleted.values()) == review["removed_character_count"]

verified_quotes = 0
for review in label_reviews:
    raw_number, raw = actual("raw", "validation", review["id"])
    processed_number, processed = actual("processed", "validation", review["id"])
    assert raw == processed
    assert raw_number == review["raw_record_number"]
    assert processed_number == review["processed_record_number"]
    assert text_sha(raw["content"]) == review["raw_text_sha256"]
    assert text_sha(processed["content"]) == review["processed_text_sha256"]
    assert review.get("source_text", review.get("text")) == raw["content"]
    for source in review.get("sources", {}).values():
        assert file_sha256(Path(source["path"])) == source["sha256"]
    if "raw_file" in review:
        assert file_sha256(Path(review["raw_file"])) == review["raw_file_sha256"]
        assert file_sha256(Path(review["processed_file"])) == review["processed_file_sha256"]
    assert [r["aspect"] for r in review["aspect_reviews"]] == list(ASPECT_COLUMNS)
    for aspect_review in review["aspect_reviews"]:
        aspect = aspect_review["aspect"]
        assert type(aspect_review["validation_label"]) is int and aspect_review["validation_label"] in CLASS_VALUES
        assert aspect_review["validation_label"] == int(raw[aspect])
        assert aspect_review["candidate_labels"]
        assert all(type(value) is int and value in CLASS_VALUES for value in aspect_review["candidate_labels"])
        proposed = aspect_review.get("suggested_label", aspect_review.get("suggested_label_for_future_reviewed_version"))
        assert proposed is None or (type(proposed) is int and proposed in CLASS_VALUES)
        for quote in aspect_review["evidence_quotes"]:
            assert quote and quote in raw["content"] and quote in processed["content"], (review["id"], aspect, quote)
            verified_quotes += 1
        training_labels = aspect_review.get("training_labels", {})
        if "training_label" in aspect_review:
            training_labels = {review["training_reference"]["id"]: aspect_review["training_label"]}
        for identity, label in training_labels.items():
            assert type(label) is int and label in CLASS_VALUES
            assert int(actual("raw", "train", identity)[1][aspect]) == label
    counterparts = review.get("training_counterparts", [])
    if "training_reference" in review:
        counterparts = [review["training_reference"]]
    for counterpart in counterparts:
        number, training_row = actual("raw", "train", counterpart["id"])
        assert number == counterpart["record_number"]
        assert text_sha(training_row["content"]) == counterpart["text_sha256"]
        assert "".join(training_row["content"].split()) == "".join(raw["content"].split())
print(f"已验证 {len(edit_review['records'])} 条编辑记录、{len(label_reviews)} 条验证样本的全部20方面，以及 {verified_quotes} 段精确引文。")


已验证 11 条编辑记录、2 条验证样本的全部20方面，以及 45 段精确引文。


## 复核结果

### 3. 全部删除记录与表情边界观察

保留文字和“任意分词器输出完全相同”是不同要求。表情文字仍完整，但不可见控制符原来可能提供分隔；这里只记录观察，不修改清洗数据。

In [4]:
# 展示每处已批准删除及其上下文，单独检查表情文字是否保留。
# 表情出现次数一致不代表任何分词器结果都一致，因此同时展示边界变化说明。
show_table(["集合 / ID", "删除数", "码点", "实义字符误删", "边界观察"], [
    (r["split"] + " / " + r["id"], r["removed_character_count"],
     ", ".join(f"{code} × {count}" for code, count in r["removed_code_point_counts"].items()),
     "发现" if r["meaningful_character_deletion_found"] else "未发现", r["boundary_risk"])
    for r in edit_review["records"]])
boundary_review = next(r for r in edit_review["records"] if r["split"] == "train" and r["id"] == "27384")
raw_text = actual("raw", "train", "27384")[1]["content"]
processed_text = actual("processed", "train", "27384")[1]["content"]
for token in ("/色", "/流泪", "/呲牙"):
    assert raw_text.count(token) == processed_text.count(token)
show_table(["原字符位置", "删除字符", "删除前局部（控制符显示为转义）", "删除后局部"], [
    (edit["raw_start"], ", ".join(edit["deleted_code_points"]),
     raw_text[max(0, edit["raw_start"]-12):edit["raw_end_exclusive"]+12].replace("\x03", "\\x03"),
     processed_text[max(0, edit["processed_position"]-12):edit["processed_position"]+12])
    for edit in boundary_review["edits"]])
display(Markdown(boundary_review["boundary_review"]))


集合 / ID,删除数,码点,实义字符误删,边界观察
train / 7649,5,U+007F × 5,未发现,none_observed
train / 12065,3,"U+001E × 1, U+001C × 2",未发现,none_observed
train / 27384,4,U+0003 × 4,未发现,low_tokenization_boundary_change
train / 56132,1,U+0001 × 1,未发现,none_observed
train / 56475,1,U+0014 × 1,未发现,none_observed
train / 80255,1,U+0001 × 1,未发现,none_observed
train / 81961,2,"U+0014 × 1, U+0001 × 1",未发现,none_observed
train / 84332,1,U+0014 × 1,未发现,none_observed
train / 96281,1,U+0014 × 1,未发现,none_observed
train / 99097,1,U+001C × 1,未发现,none_observed


原字符位置,删除字符,删除前局部（控制符显示为转义）,删除后局部
27,U+0003,球、田鸡、云吞，锅贴/色\x03/色\x03得出结论，两个字：,球、田鸡、云吞，锅贴/色/色得出结论，两个字：好
30,U+0003,鸡、云吞，锅贴/色\x03/色\x03得出结论，两个字：好吃！,田鸡、云吞，锅贴/色/色得出结论，两个字：好吃！
154,U+0003,这么好吃，太难得了/流泪\x03基本上我一个人干掉了一盒,这么好吃，太难得了/流泪基本上我一个人干掉了一盒
170,U+0003,我一个人干掉了一盒/呲牙\x03。田鸡，我的最最最爱，太,我一个人干掉了一盒/呲牙。田鸡，我的最最最爱，太


原不可见控制符兼作表情末尾分隔。删除后存在“/色/色得出结论”“/流泪基本上”，属于序列化边界变化；人工含义未变，但不得声称任意下游 tokenizer 的结果不变。应使用显式已知表情 token 识别，或在另行版本化的模型预处理中约定分隔策略。

### 4. 两条验证样本的原始冲突方面

下面逐项展示原始跨集合问题中的冲突方面；原验证标签、同组训练标签和本轮候选各自保留。候选是待统一规范的建议，不是已应用结果。

In [5]:
# 从原始冲突组重新推导争议方面，并与逐方面意见比较，防止漏展示已知分歧。
# 候选标签和置信度只用于后续裁决，当前固定验证集保持原标。
aspect_names = {"service_serving_speed": "上菜速度", "price_level": "价格水平",
                "environment_noise": "环境噪声", "dish_taste": "口味",
                "others_overall_experience": "总体体验", "dish_portion": "分量"}
conflict_rows = []
for review in label_reviews:
    conflict_names = set(review.get("original_conflicting_aspects", review.get("disputed_aspects", [])))
    fresh = next(r for r in dispute_evidence if r["id"] == review["id"])
    actual_conflicts = set().union(*(set(group["label_conflicts"]) for group in fresh["original_issue_groups"]))
    assert conflict_names == actual_conflicts
    rows = []
    for aspect_review in review["aspect_reviews"]:
        aspect = aspect_review["aspect"]
        if aspect not in conflict_names:
            continue
        training = aspect_review.get("training_labels", {})
        if "training_label" in aspect_review:
            training = {review["training_reference"]["id"]: aspect_review["training_label"]}
        candidate = "/".join(map(str, aspect_review["candidate_labels"]))
        row = (aspect_names.get(aspect, aspect), aspect_review["validation_label"],
               ", ".join(identity + ":" + str(label) for identity, label in training.items()),
               candidate + " (" + aspect_review["confidence"] + ")",
               "；".join(aspect_review["evidence_quotes"]))
        rows.append(row)
        conflict_rows.append((review["id"], aspect, row))
    assert len(rows) == len(actual_conflicts)
    display(Markdown("**验证集 ID " + review["id"] + "**"))
    show_table(["方面", "原验证标签", "同文训练 ID:标签", "本轮候选 / 置信度", "精确原文证据"], rows)
    if "assessment_summary" in review:
        display(Markdown("\n\n".join(review["assessment_summary"][aspect] for aspect in ("price_level", "dish_portion"))))
    else:
        display(Markdown("\n\n".join(r["reason"] for r in review["aspect_reviews"]
                                    if r["aspect"] in actual_conflicts)))
print("原始冲突方面合计：", len(conflict_rows))


原始冲突方面合计： 7


**验证集 ID 13036**

方面,原验证标签,同文训练 ID:标签,本轮候选 / 置信度,精确原文证据
上菜速度,-2,45884:-1,-1 (high),我们没主食，点了米饭，催了好几次才上来。
价格水平,0,45884:-1,-1/0 (medium),【花蛤炸弹鸡】RMB 68；估计是因为有花蛤的原因，不然这么一份要68块还真不便宜
环境噪声,0,45884:-2,-2 (high),
口味,0,45884:1,0/1 (medium_high),味道还可以吧；这么光鸡肉吃起来有一些单调。；这里可能会让你失望了。沾着黄豆酱吃，也还不错吧~
总体体验,0,45884:-1,-1 (high),就是那个服务员让我这顿饭吃的实在很窝火；敢情我今天不是来吃饭的，是来让你们安排的。


明确多次催促才上米饭，上菜速度负面；原验证标签未提及与直接证据不符。

有明确价格及不便宜的评价，倾向负面；但该评价位于不然条件句中，作者也给出花蛤解释，原中性仍有解释空间。

全文没有嘈杂、安静、吵闹或音量描述；临马路不等于存在噪音评价，服务员喋喋不休属于服务互动。

有还可以、不错的轻度褒义，也有单调、失望的负面；多菜品混合评价，原中性合理，训练正面不是唯一正确答案。

对整顿饭的体验明确表达窝火，被服务问题主导；原中性与全文结尾的整体负面表态冲突。

**验证集 ID 14365**

方面,原验证标签,同文训练 ID:标签,本轮候选 / 置信度,精确原文证据
价格水平,1,"30208:0, 75142:0",0/1 (medium),56一斤，2.2斤；一共170元；性价比还是很不错的。
分量,-2,"30208:-2, 75142:0",0/1 (medium),我挑了一条2.2斤的黑鱼；还加了一份娃娃菜；两个人吃有点撑了。


验证1 vs 训练0/0。明确报价但没有绝对贵便宜判断，正面的性价比不自动等于价格便宜；按显性情感口径倾向0，保留0/1口径歧义。

验证-2 vs 训练-2/0。两个人吃有点撑明确提及分量，-2最值得复核；倾向0，量足为正面规范下可为1。

## 结论与后续事项

机械证据证明的是当前原始 CSV 到清洗 CSV 的可观测变化；它不能恢复上游已经损坏的文本，也不能证明所有标签正确。语义建议保留原文证据、候选和不确定性，固定验证标签仍未修改。

本轮未开展全量语义重标、模糊语义去重或时间趋势分析；数据没有可用评论时间。

In [6]:
# 将逐条语义复核汇总与全量机械对账相互核对，最后复查全部受保护文件的摘要。
# 本单元只验证历史证据是否仍适用，不把助手建议自动转成标签修改。
summary = edit_review["summary"]
assert summary["edited_records_reviewed"] == len(edit_evidence)
assert summary["removed_characters_reviewed"] == audit["removed_character_count"]
assert summary["removed_code_point_counts"] == audit["removed_characters"]
assert summary["meaningful_character_deletions_found"] == sum(r["meaningful_character_deletion_found"] for r in edit_review["records"])
# 确认上述只读核验结束后，原始数据、派生数据和既有报告仍与受保护摘要一致。
for filename, expected in audit["file_sha256"].items():
    assert file_sha256(Path(filename)) == expected
print(f"全量对账通过：{sum(audit['retained_counts'].values()):,} 条保留评论、{audit['label_cells_compared']:,} 个标签单元格。")
print(f"逐条语义复核：{summary['edited_records_reviewed']} 条文本、{summary['removed_characters_reviewed']} 个删除字符；记录的实义字符误删数：{summary['meaningful_character_deletions_found']}。")
print(f"两条验证样本共复核 {sum(len(r['aspect_reviews']) for r in label_reviews)} 个方面，原始冲突方面 {len(conflict_rows)} 个；建议尚未应用。")
print("原始数据、清洗数据、隔离数据及既有处理报告摘要均保持不变。")


全量对账通过：119,969 条保留评论、2,399,380 个标签单元格。
逐条语义复核：11 条文本、21 个删除字符；记录的实义字符误删数：0。
两条验证样本共复核 40 个方面，原始冲突方面 7 个；建议尚未应用。
原始数据、清洗数据、隔离数据及既有处理报告摘要均保持不变。
